# View Rafael's DES Y3 shear maps

A shear map records a small distortion of galaxy images. Two components
specify that distortion relative to a coordinate basis. This notebook
inspects **two files only** and previews the DES footprint and shear values;
it does not measure correlations or estimate a covariance.

Rafael confirmed NSIDE 512 and that `indices.npy` lists the footprint pixels.
Read the [folder guide](README.md) for conventions still needed. The map
preview is optional: missing coordinates or an unconfirmed ordering must
not be replaced by guessed sky positions.

In [1]:
from pathlib import Path
import numpy as np
from matplotlib import pyplot as plt

# Open this notebook with maps_cov_512_v2 as the working directory.
folder = Path.cwd()
filenames = [
    "ShearMaps_DESY3_run_0000_ncut0_nreal0.npy",
    "ShearMaps_DESY3_run_0000_ncut0_nreal1.npy",
]
nside = 512
source_bin = 0  # First array slot; confirm the physical bin order with Rafael.
nested = None  # Set False for RING or True for NESTED after confirmation.
coordinate_frame = None  # Set "equatorial" only if confirmed by the producer.

maps = []
for filename in filenames:
    # Memory mapping reads the requested slices without loading the archive.
    shear = np.load(folder / filename, mmap_mode="r", allow_pickle=False)
    if shear.ndim != 3 or shear.shape[:2] != (4, 2):
        raise ValueError("Inspect the file: expected four bins and two components.")
    maps.append(shear)
    sample = shear[:, :, :4096]
    print(filename)
    print("shape:", shear.shape, "dtype:", shear.dtype)
    print("first 4096 pixels finite:", np.isfinite(sample).all())
    print("first five values in bin 0, component 0:", shear[0, 0, :5])

ShearMaps_DESY3_run_0000_ncut0_nreal0.npy
shape: (4, 2, 369190) dtype: float64
first 4096 pixels finite: True
first five values in bin 0, component 0: [-0.02587572 -0.01322676  0.00673684 -0.04449044  0.00759723]
ShearMaps_DESY3_run_0000_ncut0_nreal1.npy
shape: (4, 2, 369190) dtype: float64
first 4096 pixels finite: True
first five values in bin 0, component 0: [-0.00342078  0.03296318 -0.04814199 -0.06067539  0.02829669]


## Match array positions to sky pixels

The integer `j` in `maps[0][0, 0, j]` is a position in the saved list.
Its sky location is pixel `indices[j]`. Preserve that order. The checks
below establish basic consistency of a common pixel list; they cannot
establish the correct ordering, coordinate basis or weights.

A missing `indices.npy` is expected while the download is incomplete. If it
has several lists, one per footprint, select the documented list for `ncut0`
instead of flattening it.

In [2]:
indices = None
index_path = folder / "indices.npy"
if not index_path.is_file():
    print("indices.npy is not local yet; footprint plots are pending.")
else:
    indices = np.load(index_path, mmap_mode="r", allow_pickle=False)
    if indices.ndim != 1 or indices.dtype.kind not in "iu":
        raise ValueError("Inspect indices.npy: this example expects one integer list.")
    if any(shear.shape[-1] != indices.size for shear in maps):
        raise ValueError("The pixel list must match the last axis of both maps.")
    if np.any(indices < 0) or np.any(indices >= 12 * nside**2):
        raise ValueError("A pixel index is outside the NSIDE 512 full sky.")
    if np.unique(indices).size != indices.size:
        raise ValueError("The footprint pixel list contains repeated indices.")
    print("Retained footprint pixels:", indices.size)
    print("First pixel indices:", indices[:5])

indices.npy is not local yet; footprint plots are pending.


## Plot the footprint and two shear maps

The plot uses at most 30,000 retained positions to keep the preview quick.
This is a display choice, not a thinning rule for science measurements.
The panels share a symmetric color range, so colors have the same meaning
for both realizations. Values beyond the displayed 99th percentile saturate
the colors; the data are unchanged. A gap in this scatter preview need not
be a hole in the complete mask.

The array slots are labeled “component 0/1” until their physical signs and
basis are confirmed. Use the producer's convention when constructing a
TreeCorr catalog; a visually plausible map cannot validate that choice.

In [3]:
hp = None
try:
    import healpy as hp
except ImportError:
    print("The footprint preview requires healpy in this notebook environment.")

if indices is None or nested is None or hp is None:
    print("Plot pending: supply indices.npy, confirm nested, and provide healpy.")
else:
    # Read two small slices, never all maps in the directory.
    positions = np.linspace(
        start=0,
        stop=indices.size - 1,
        num=min(30000, indices.size),
        dtype=int,
    )
    longitude, latitude = hp.pix2ang(
        nside=nside,
        ipix=indices[positions],
        nest=nested,
        lonlat=True,
    )
    longitude = (longitude + 180.0) % 360.0 - 180.0
    xlabel = "RA, wrapped [deg]" if coordinate_frame == "equatorial" else "Longitude [deg]"
    ylabel = "Dec [deg]" if coordinate_frame == "equatorial" else "Latitude [deg]"
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.scatter(longitude, latitude, s=1, color="#0072B2", rasterized=True)
    ax.set(xlabel=xlabel, ylabel=ylabel, title="DES footprint: sampled retained pixels")
    plt.show()

    values = np.stack([shear[source_bin][:, positions] for shear in maps])
    limit = np.quantile(np.abs(values), 0.99)
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
    for realization in range(2):
        for component in range(2):
            ax = axes[realization, component]
            points = ax.scatter(
                longitude,
                latitude,
                c=values[realization, component],
                s=2,
                cmap="RdBu_r",
                vmin=-limit,
                vmax=limit,
                rasterized=True,
            )
            ax.set(
                xlabel=xlabel,
                ylabel=ylabel,
                title=f"nreal{realization}, bin slot {source_bin}, component {component}",
            )
    fig.colorbar(points, ax=axes.ravel().tolist(), label="Shear (dimensionless)")
    plt.show()

The footprint preview requires healpy in this notebook environment.
Plot pending: supply indices.npy, confirm nested, and provide healpy.


## What these pictures establish

They let us inspect the footprint and compare shear values at the same sky
positions. They do **not** show that the two files contain independent
large-scale structure, that the signs match the correlation estimator, or
that NSIDE 512 resolves the smallest aperture used in the analysis.

Proceed to the [covariance notebook](../EXAMPLE_COVARIANCE_PME.ipynb) for
the difference between a map, a measured joint vector and an independent
simulation. Generated plots and vectors belong in `../chains/`; the
simulation files stay out of Git.